# Load

In [8]:
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
import matplotlib as mpl
from matplotlib.colors import TwoSlopeNorm
from matplotlib import gridspec
from matplotlib.colors import TwoSlopeNorm
import matplotlib.pyplot as plt
from pathlib import Path
import json
import numpy as np
import pandas as pd
from cmcrameri import cm

In [9]:
# mpl.use("Cairo")
mpl.rcParams["pdf.compression"] = 0
mpl.rcParams["savefig.transparent"] = False

In [10]:

%matplotlib inline
plt.rc('font',family='arial')
plt.rcParams.update({"font.size":6})
font= fm.FontProperties(family='arial')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# cmcrameri.cm.show_cmaps()



# Settings

In [11]:
classifiacation_palette = cm.lipariS.colors
# classifiacation_palette = cm.actonS.colors[[i for i in range(10) if i != 1],:]


MODEL_COLOR = classifiacation_palette[0]
OBS_COLOR = classifiacation_palette[2]

MODEL_MARKER = "D"
OBS_MARKER = "o"

# Publication figure

## Functions

In [12]:

def load_plot_data(data_dir):
    data_dir = Path(data_dir)

    with open(data_dir / "metadata.json", encoding="utf-8") as file:
        metadata = json.load(file)

    heatmap = pd.read_csv(
        data_dir / "reflectivity_deviation_heatmap.csv"
    )

    anomaly = pd.read_csv(
        data_dir / "sedimentation_anomaly.csv"
    )

    correlation = pd.read_csv(
        data_dir / "peak_ice_fraction_correlation.csv"
    )

    return {
        "metadata": metadata,
        "heatmap": heatmap,
        "anomaly": anomaly,
        "correlation": correlation,
    }


def prepare_heatmap(dataset):
    """Convert long-format heatmap data back into a 2-D array."""
    metadata = dataset["metadata"]
    heatmap = dataset["heatmap"]

    rel_times = np.asarray(
        metadata["settings"]["rel_times_min"],
        dtype=int,
    )

    # Preserve the order in the CSV
    level_labels = (
        heatmap["vertical_level"]
        .drop_duplicates()
        .tolist()
    )

    matrix = (
        heatmap
        .pivot(
            index="vertical_level",
            columns="rel_time_min",
            values="reflectivity_deviation_dbz",
        )
        .reindex(
            index=level_labels,
            columns=rel_times,
        )
        .to_numpy()
    )

    return matrix, level_labels, rel_times


def format_heatmap(
    ax,
    matrix,
    level_labels,
    rel_times,
    title,
    norm,
    y_ticks =True,
    hatch = (True,True)
):
    image = ax.imshow(
        matrix,
        aspect="auto",
        cmap=cm.vik,
        norm=norm,
        origin="upper",
        rasterized=True
    )

    ax.set_xticks(np.arange(len(rel_times)))
    ax.set_xticklabels(rel_times)
    if y_ticks:
        # ax.set_yticks(np.arange(len(level_labels)))
        # ax.set_yticklabels(level_labels)
        ax.set_yticks([-0.5,0.5,1.5,2.5])
        ax.set_yticklabels([0,-600,-1440,-2280])
    else:
        ax.set_yticks([])

    ax.set_xlabel("Time relative to glaciation peak [min]")
    ax.set_title(title)

    # Hatch the -45 and +45 minute columns
    for i, selected_time in enumerate((-45, 45)):
        matching_indices = np.where(
            rel_times == selected_time
        )[0]
        if hatch[i]:
            for index in matching_indices:
                ax.axvspan(
                    index - 0.5,
                    index + 0.5,
                    facecolor="none",
                    edgecolor="0.3",
                    hatch="/////",
                    linewidth=0,
                    zorder=5,
                )

    return image


In [13]:
def make_combined_figure(
    model_dir,
    observations_dir,
    output_file=None,
):
    # --------------------------------------------------------------
    # Load data
    # --------------------------------------------------------------
    model = load_plot_data(model_dir)
    observations = load_plot_data(observations_dir)

    (
        model_heatmap,
        model_levels,
        model_times,
    ) = prepare_heatmap(model)

    (
        obs_heatmap,
        obs_levels,
        obs_times,
    ) = prepare_heatmap(observations)

    model_confidence = (
        100
        * model["metadata"]["settings"]["confidence_level"]
    )

    obs_confidence = (
        100
        * observations["metadata"]["settings"]["confidence_level"]
    )

    same_confidence = np.isclose(
        model_confidence,
        obs_confidence,
    )

    same_levels = list(model_levels) == list(obs_levels)

    # --------------------------------------------------------------
    # Figure layout
    # --------------------------------------------------------------
    fig = plt.figure(
        figsize=(5.501, 3.5),
        constrained_layout=True,
    )

    gs = gridspec.GridSpec(
        3,
        3,
        figure=fig,
        height_ratios=[1, 1.15, 1],
        width_ratios=[1, 1, 0.045],
    )

    # Combined correlation panel
    ax_corr = fig.add_subplot(gs[0, :2])

    # Side-by-side heatmaps
    ax_heat_model = fig.add_subplot(gs[1, 1])
    ax_heat_obs = fig.add_subplot(gs[1, 0])
    ax_colorbar = fig.add_subplot(gs[1, 2])

    # Combined sedimentation anomaly panel
    ax_anom = fig.add_subplot(gs[2, :2])

    # --------------------------------------------------------------
    # 1. Combined correlation panel
    # --------------------------------------------------------------
    model_corr = model["correlation"]
    obs_corr = observations["correlation"]

    
    ax_corr.plot(
        obs_corr["rel_time_min"],
        obs_corr["corr_with_peak"],
        marker=OBS_MARKER,
        markersize=3,
        linewidth=1.2,
        color=OBS_COLOR,
        label="Observations",
        zorder=5,
    )

    ax_corr.fill_between(
        obs_corr["rel_time_min"],
        obs_corr["ci_low"],
        obs_corr["ci_high"],
        color=OBS_COLOR,
        alpha=0.20,
        linewidth=0,
        zorder=3,
    )

    ax_corr.plot(
        model_corr["rel_time_min"],
        model_corr["corr_with_peak"],
        marker=MODEL_MARKER,
        markersize=3,
        linewidth=1.2,
        color=MODEL_COLOR,
        label="ICON",
        zorder=4,
        linestyle=":"
    )

    ax_corr.fill_between(
        model_corr["rel_time_min"],
        model_corr["ci_low"],
        model_corr["ci_high"],
        color=MODEL_COLOR,
        alpha=0.20,
        linewidth=0,
        zorder=2,
    )


    ax_corr.axhline(
        0,
        color="k",
        linewidth=1,
        zorder=6,
    )

    ax_corr.axhspan(
        -0.1,
        0.1,
        facecolor="none",
        edgecolor="0.3",
        hatch="/////",
        linewidth=0,
        zorder=6,
        label="Below 0.1",
    )

    all_corr_times = np.concatenate([
        model_corr["rel_time_min"].to_numpy(),
        obs_corr["rel_time_min"].to_numpy(),
    ])

    corr_xmin = np.nanmin(all_corr_times)
    corr_xmax = np.nanmax(all_corr_times)

    corr_xticks = np.arange(
        np.floor(corr_xmin / 15) * 15,
        np.ceil(corr_xmax / 15) * 15 + 15,
        15,
    )
    corr_xlabels = [
        r"$t_{\mathrm{peak}}$" if tick == 0 else f"{tick:+g}"
        for tick in corr_xticks
    ]

    ax_corr.set_xticks(corr_xticks,corr_xlabels)
    ax_corr.set_xlim(-45 - 7.5, 45 + 7.5)
    ax_corr.set_ylim(-0.1, 1.05)

    ax_corr.set_xlabel(
        "Time relative to glaciation peak [min]"
    )
    ax_corr.set_ylabel("Correlation with\n peak Ice Fracton")
    # ax_corr.set_title(
    #     "Correlation with peak ice fraction"
    # )

    # ax_corr.grid(alpha=0.3)

    if same_confidence:
        legend_title = (
            f"Shading: {model_confidence:g}% bootstrap CI"
        )
    else:
        legend_title = (
            f"Model CI: {model_confidence:g}%; "
            f"observations CI: {obs_confidence:g}%"
        )
    legend_title=""
    ax_corr.legend(
        loc="upper right",
        title=legend_title,
    )

    # --------------------------------------------------------------
    # 2. Side-by-side heatmaps
    # --------------------------------------------------------------
    heatmap_norm = TwoSlopeNorm(
        vmin=-3,
        vcenter=0,
        vmax=3,
    )

    model_image = format_heatmap(
        ax=ax_heat_model,
        matrix=model_heatmap,
        level_labels=model_levels,
        rel_times=model_times,
        title="ICON",
        norm=heatmap_norm,
        y_ticks=False,
        hatch=(True,False)

    )

    format_heatmap(
        ax=ax_heat_obs,
        matrix=obs_heatmap,
        level_labels=obs_levels,
        rel_times=obs_times,
        title="Observations",
        norm=heatmap_norm,
    )

    ax_heat_obs.set_ylabel("Distance from top [m]")

    # Avoid duplicated vertical labels when levels are identical
    ax_heat_model.tick_params(
            axis="y",
            labelleft=False,
        )


    colorbar = fig.colorbar(
        model_image,
        cax=ax_colorbar,
    )

    colorbar.set_label("Deviation from\nlevel mean[dBZ]")
    colorbar.set_ticks([-3, -1.5, 0, 1.5, 3])

    # A title applying to both heatmaps
    # fig.text(
    #     0.5,
    #     0.62,
    #     "Radar reflectivity deviation from level mean",
    #     ha="center",
    #     va="bottom",
    #     fontproperties=font,
    # )

        # --------------------------------------------------------------
    # 3. Combined sedimentation anomaly panel
    # --------------------------------------------------------------
    model_anom = (
        model["anomaly"]
        .sort_values("rel_time_min")
        .reset_index(drop=True)
    )

    obs_anom = (
        observations["anomaly"]
        .sort_values("rel_time_min")
        .reset_index(drop=True)
    )

    # Model values
    model_time = model_anom["rel_time_min"].to_numpy()
    model_value = model_anom[
        "sedimentation_anomaly_dbz"
    ].to_numpy()
    model_ci_low = model_anom["ci_low_dbz"].to_numpy()
    model_ci_high = model_anom["ci_high_dbz"].to_numpy()

    # Observation values
    obs_time = obs_anom["rel_time_min"].to_numpy()
    obs_value = obs_anom[
        "sedimentation_anomaly_dbz"
    ].to_numpy()
    obs_ci_low = obs_anom["ci_low_dbz"].to_numpy()
    obs_ci_high = obs_anom["ci_high_dbz"].to_numpy()


    # Observation line and shaded confidence interval
    ax_anom.fill_between(
        obs_time,
        obs_ci_low,
        obs_ci_high,
        color=OBS_COLOR,
        alpha=0.20,
        linewidth=0,
        zorder=3,
    )

    ax_anom.plot(
        obs_time,
        obs_value,
        marker=OBS_MARKER,
        linewidth=1.5,
        markersize=3.5,
        color=OBS_COLOR,
        label="Observations",
        zorder=5,
    )
    
    # Model line and shaded confidence interval
    ax_anom.fill_between(
        model_time,
        model_ci_low,
        model_ci_high,
        color=MODEL_COLOR,
        alpha=0.20,
        linewidth=0,
        zorder=2,
    )

    ax_anom.plot(
        model_time,
        model_value,
        marker=MODEL_MARKER,
        linewidth=1.5,
        markersize=3.5,
        color=MODEL_COLOR,
        label="ICON",
        zorder=4,
        linestyle=":"
    )



    ax_anom.axhline(
        0,
        color="k",
        linewidth=1,
        alpha=0.5,
        zorder=1,
    )

    all_anomaly_times = np.concatenate([
        model_time,
        obs_time,
    ])

    anomaly_xticks = np.arange(
        np.floor(np.nanmin(all_anomaly_times) / 15) * 15,
        np.ceil(np.nanmax(all_anomaly_times) / 15) * 15 + 15,
        15,
    )
    anomaly_xlabels = [
        r"$t_{\mathrm{peak}}$" if tick == 0 else f"{tick:+g}"
        for tick in anomaly_xticks
    ]

    ax_anom.set_xticks(anomaly_xticks,anomaly_xlabels)
    ax_anom.set_xlim(-45 - 7.5, 45 + 7.5)

    ax_anom.set_ylim(-6, 6)
    ax_anom.set_yticks(np.arange(-6, 7, 3))

    ax_anom.set_xlabel(
        "Time relative to glaciation peak [min]"
    )
    ax_anom.set_ylabel("Sedimentation\nanomaly [dBZ]")
    # ax_anom.set_title("Sedimentation anomaly")

    if same_confidence:
        anomaly_legend_title = (
            f"Shading: {model_confidence:g}% bootstrap CI"
        )
    else:
        anomaly_legend_title = (
            f"ICON CI: {model_confidence:g}%; "
            f"observations CI: {obs_confidence:g}%"
        )
    anomaly_legend_title=""
    ax_anom.legend(
        loc="upper right",
        title=anomaly_legend_title,
    )
    # Make ticks lines smaller
    for ax in (
        ax_corr,
        ax_heat_model,
        ax_heat_obs,
        ax_anom,
    ):
        ax.tick_params(
            axis="both",
            which="major",
            length=2.5,      # tick-line length
            width=1,     # tick-line thickness
        )

    # --------------------------------------------------------------
    # Save
    # --------------------------------------------------------------
    if output_file is not None:
        output_file = Path(output_file)
        output_file.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig.savefig(
            output_file,
            dpi=300,
            bbox_inches="tight",
            transparent=False
        )
        fig.savefig(
            output_file.with_suffix(".svg"),
            dpi=300,
            bbox_inches="tight",
            transparent=False
        )

    return fig

## Figure

In [14]:
base_dir = Path(
    "/Volumes/University_stuff/Code_Projects/MSGI/plotting/"
    "Publication_figures/data"
)

fig = make_combined_figure(
    model_dir=base_dir / "model",
    observations_dir=base_dir / "observations",
    output_file=base_dir / "model_observations_combined.pdf",
)

plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/University_stuff/Code_Projects/MSGI/plotting/Publication_figures/data/model/metadata.json'